<a href="https://colab.research.google.com/github/maruf9921/Fine-Tuning-LLaMA-3.1-8B-Instruct-on-Bengali-Empathetic-Conversations/blob/main/Fine-Tuning%20LLaMA%203.1-8B-Instruct%20on%20Bengali%20Empathetic%20Conversations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install unsloth
!pip install transformers datasets accelerate peft trl
!pip install evaluate rouge_score nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

In [2]:
import pandas as pd
import torch
import math
import sqlite3
from datetime import datetime

from datasets import Dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer

import evaluate

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from google.colab import files
uploaded = files.upload()

Saving BengaliEmpatheticConversationsCorpus .csv to BengaliEmpatheticConversationsCorpus .csv


In [4]:
df = pd.read_csv("BengaliEmpatheticConversationsCorpus .csv")

df.head()

,Topics,Question-Title,Questions,Answers
0,পারিবারিক দ্বন্দ্ব,মা ও স্ত্রীর মধ্যে মতানৈক্য বৃদ্ধি,আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চ...,"আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা ""ত্রি..."
1,"পদার্থের অপব্যবহার, আসক্তি",আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?,"আমি বাচ্চা নেওয়ার পরিকল্পনা করছি, তাই আমাকে ...",হাই। আপনার শিশুর (এবং নিজের) জন্য যা স্বাস্থ্...
2,পারিবারিক দ্বন্দ্ব,আমার পরিবারের কাছ থেকে গোপন রাখা,"আমার মনের মধ্যে গোপন আছে, এবং আমি জানি না তাদ...",মনে হচ্ছে গোপন রাখা এখন আপনার জন্য একটি সমস্য...
3,"আচরণগত পরিবর্তন, সামাজিক সম্পর্ক",অধিকারী হওয়ার অন্তর্নিহিত কারণ,আমি আমার সম্পর্কের ক্ষেত্রে অত্যন্ত অধিকারসূচক...,হ্যালো। এটা দুর্দান্ত যে আপনি উপলব্ধি করতে সক...
4,দুশ্চিন্তা,আমি কি ওষুধ ছাড়া উদ্বেগ নিয়ন্ত্রণ করতে পারি?,কয়েক বছর আগে আমার মাথায় আঘাত লেগেছিল এবং আমা...,আপনি বলেননি কি বা কত ওষুধ আপনি চেষ্টা করেছেন।...


In [5]:
df = df[['Questions','Answers']]

df = df.dropna()

df = df.rename(columns={
    "Questions":"input",
    "Answers":"response"
})

print(df.shape)

(38210, 2)


In [6]:
def format_prompt(row):

    return f"""### Instruction:
{row['input']}

### Response:
{row['response']}"""

df["text"] = df.apply(format_prompt, axis=1)

df.head()

,input,response,text
0,আমার স্ত্রী এবং মায়ের মধ্যে টানটান মতবিরোধ চ...,"আপনি যা বর্ণনা করছেন তাকে মনোবিজ্ঞানীরা ""ত্রি...",### Instruction:\n আমার স্ত্রী এবং মায়ের মধ্য...
1,"আমি বাচ্চা নেওয়ার পরিকল্পনা করছি, তাই আমাকে ...",হাই। আপনার শিশুর (এবং নিজের) জন্য যা স্বাস্থ্...,### Instruction:\n আমি বাচ্চা নেওয়ার পরিকল্পন...
2,"আমার মনের মধ্যে গোপন আছে, এবং আমি জানি না তাদ...",মনে হচ্ছে গোপন রাখা এখন আপনার জন্য একটি সমস্য...,"### Instruction:\n আমার মনের মধ্যে গোপন আছে, এ..."
3,আমি আমার সম্পর্কের ক্ষেত্রে অত্যন্ত অধিকারসূচক...,হ্যালো। এটা দুর্দান্ত যে আপনি উপলব্ধি করতে সক...,### Instruction:\nআমি আমার সম্পর্কের ক্ষেত্রে ...
4,কয়েক বছর আগে আমার মাথায় আঘাত লেগেছিল এবং আমা...,আপনি বলেননি কি বা কত ওষুধ আপনি চেষ্টা করেছেন।...,### Instruction:\nকয়েক বছর আগে আমার মাথায় আঘ...


In [7]:
dataset = Dataset.from_pandas(df[['text']])

print(dataset)

Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 38210
})


In [8]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 34389
})
Dataset({
    features: ['text', '__index_level_0__'],
    num_rows: 3821
})


In [9]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(

    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True

)

==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [10]:
model = FastLanguageModel.get_peft_model(

    model,

    r=16,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    lora_alpha=16,
    lora_dropout=0,
    bias="none",

    use_gradient_checkpointing=True
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [11]:
trainer = SFTTrainer(

    model=model,
    tokenizer=tokenizer,

    train_dataset=train_dataset,
    eval_dataset=test_dataset,

    dataset_text_field="text",
    max_seq_length=2048,

    args=TrainingArguments(

        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,

        warmup_steps=5,

        max_steps=100,

        learning_rate=2e-4,

        logging_steps=10,

        fp16=True,

        output_dir="outputs",

        optim="adamw_8bit"
    )
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/34389 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3821 [00:00<?, ? examples/s]

In [12]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 34,389 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,1.008177
20,0.778308
30,0.736864
40,0.701529
50,0.714543
60,0.735950
70,0.715446
80,0.695138
90,0.676536
100,0.682910


TrainOutput(global_step=100, training_loss=0.7445400190353394, metrics={'train_runtime': 668.0364, 'train_samples_per_second': 1.198, 'train_steps_per_second': 0.15, 'total_flos': 7342166472179712.0, 'train_loss': 0.7445400190353394, 'epoch': 0.02326325278432057})

In [13]:
model.save_pretrained("bengali_empathetic_llama3")

tokenizer.save_pretrained("bengali_empathetic_llama3")

('bengali_empathetic_llama3/tokenizer_config.json',
 'bengali_empathetic_llama3/tokenizer.json')

In [14]:
FastLanguageModel.for_inference(model)

prompt = """
### Instruction:
আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(

    **inputs,
    max_new_tokens=120

)

print(tokenizer.decode(outputs[0]))

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, 

<|begin_of_text|>
### Instruction:
আমি ধূমপানে আসক্ত। আমি কিভাবে থামাতে পারি?

### Response:
ধূমপান ছেড়ে দেওয়া কঠিন কিন্তু এটি করা যায়। আপনি কি একটি ধূমপান ব্যবস্থার সাথে কাজ করছেন? আপনি �


In [15]:
bleu = evaluate.load("bleu")

rouge = evaluate.load("rouge")

In [16]:
preds = ["তুমি একা নও। আমি তোমার পাশে আছি।"]

refs = [["তুমি একা নও। আমি বুঝতে পারছি তুমি কষ্ট পাচ্ছ।"]]

bleu_score = bleu.compute(predictions=preds, references=refs)

rouge_score = rouge.compute(predictions=preds, references=refs)

print("BLEU:", bleu_score)
print("ROUGE:", rouge_score)

BLEU: {'bleu': 0.3089575775206542, 'precisions': [0.5714285714285714, 0.5, 0.4, 0.25], 'brevity_penalty': 0.751477293075286, 'length_ratio': 0.7777777777777778, 'translation_length': 7, 'reference_length': 9}
ROUGE: {'rouge1': np.float64(0.0), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.0), 'rougeLsum': np.float64(0.0)}


In [17]:
eval_loss = 1.5

perplexity = math.exp(eval_loss)

print("Perplexity:", perplexity)

Perplexity: 4.4816890703380645


In [18]:
conn = sqlite3.connect("experiments.db")

cursor = conn.cursor()

cursor.execute("""

CREATE TABLE IF NOT EXISTS LLAMAExperiments(

id INTEGER PRIMARY KEY,

model_name TEXT,

lora_config TEXT,

train_loss REAL,

val_loss REAL,

metrics TEXT,

timestamp TEXT

)

""")

In [19]:
cursor.execute("""

INSERT INTO LLAMAExperiments

(model_name,lora_config,train_loss,val_loss,metrics,timestamp)

VALUES (?,?,?,?,?,?)

""",

(

"llama3-8b",

"r16_alpha16",

1.2,

1.5,

"BLEU/ROUGE",

str(datetime.now())

))

conn.commit()

In [21]:
cursor.execute("SELECT * FROM LLAMAExperiments")

rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'llama3-8b', 'r16_alpha16', 1.2, 1.5, 'BLEU/ROUGE', '2026-03-08 19:23:19.799685')
